# Gait Dashboard — Interactive Plotly Charts

All charts are fully interactive — hover for exact values, click legend items
to toggle feet, zoom/pan, and download any chart as a PNG using the toolbar.

**Every figure shows Midstance Entropy (left panel) and Loading Rate (right panel) side-by-side.**

| Cell | Chart type |
|---|---|
| 5  | Line graph — metric vs walk time |
| 6  | Scatter plot — value vs step number |
| 7  | Bar chart — per-step magnitude interleaved |
| 8  | Rolling mean ± 1 SD band |
| 9  | Box plot + strip (raw points overlaid) |
| 10 | Histogram with mean lines |
| 11 | Cumulative distribution (ECDF) |
| 12 | Left–Right symmetry (|R−L| per matched pair) |
| 13 | Violin plot |
| 14 | Cross-metric scatter — entropy vs loading rate |

## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import entropy as scipy_entropy, mannwhitneyu
from scipy.signal import find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print(f'Plotly ready')

Plotly ready


## Cell 2 — Configuration

In [2]:
CSV_PATH = 'Kristian_dry_0425.csv'

PRESSURE_COLS     = [f'pressure_{i:02d}' for i in range(1, 13)]
HEEL_SENSORS      = ['pressure_08', 'pressure_11']
MAX_ENTROPY       = np.log2(12)
SAMPLE_RATE_MS    = 4

# Stance detection
STANCE_PERCENTILE = 15
MIN_STANCE_SAMP   = 20
MAX_STANCE_SAMP   = 500
MS_START, MS_END  = 1/3, 2/3

# Loading rate (v7)
SMOOTH_WIN    = 11
SMOOTH_POLY   = 2
BIG_PEAK_DIST = 50
BIG_PEAK_PROM = 200
BIG_PEAK_HT   = 500
HILL_WINDOW   = 10
MIN_DP        = 5

# Colours — consistent across every chart
C_RIGHT = '#1F77B4'
C_LEFT  = '#FF7F0E'

# Plotly layout defaults applied to every figure
LAYOUT = dict(
    template='plotly_white',
    font=dict(family='Arial', size=12),
    hoverlabel=dict(bgcolor='white', font_size=12),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(t=80, b=50, l=60, r=40),
)

print('Config ready')
print(f'  Right foot colour: {C_RIGHT}')
print(f'  Left  foot colour: {C_LEFT}')

Config ready
  Right foot colour: #1F77B4
  Left  foot colour: #FF7F0E


## Cell 3 — Load & Extract Both Metrics

In [3]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
raw = raw[raw['corrupt'] == 0].copy()
raw['timestamp']      = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['total_pressure'] = raw[PRESSURE_COLS].sum(axis=1)

# ── Stance detection ──────────────────────────────────────────────────────────
def detect_stances(sdf):
    sig = sdf['total_pressure'].values
    thr = np.percentile(sig, STANCE_PERCENTILE)
    in_s = sig >= thr
    wins, i = [], 0
    while i < len(in_s):
        if in_s[i]:
            s = i
            while i < len(in_s) and in_s[i]: i += 1
            d = i - s
            if MIN_STANCE_SAMP <= d <= MAX_STANCE_SAMP:
                wins.append({'start': s, 'end': i, 'duration': d})
        else:
            i += 1
    return wins

# ── Entropy extraction ────────────────────────────────────────────────────────
def extract_entropy(sdf, foot_label):
    wins = detect_stances(sdf)
    t0   = sdf['timestamp'].iloc[0]
    recs = []
    for idx, win in enumerate(wins):
        s, e, d = win['start'], win['end'], win['duration']
        ms_s = s + int(d * MS_START)
        ms_e = s + int(d * MS_END)
        if ms_e <= ms_s: continue
        mp  = sdf.iloc[ms_s:ms_e][PRESSURE_COLS].mean().values.astype(float)
        tot = mp.sum()
        if tot < 10: continue
        p = np.clip(mp / tot, 1e-10, 1.0)
        H = scipy_entropy(p, base=2)
        t = (sdf.iloc[(ms_s + ms_e) // 2]['timestamp'] - t0) / 1000
        recs.append({
            'step': idx, 'foot': foot_label, 'time_s': round(float(t), 3),
            'evenness': round(H / MAX_ENTROPY, 6),
            'H_bits': round(H, 6),
            'unevenness': round(1 - H / MAX_ENTROPY, 6),
            'stance_ms': d * SAMPLE_RATE_MS,
        })
    return pd.DataFrame(recs)

# ── Loading rate extraction (v7) ──────────────────────────────────────────────
def find_step_windows(sdf):
    y = sum(pd.to_numeric(sdf[s], errors='coerce').fillna(0).to_numpy(dtype=float)
            for s in HEEL_SENSORS if s in sdf.columns)
    n  = len(y)
    sw = min(SMOOTH_WIN, n - (0 if n % 2 == 1 else 1))
    if sw % 2 == 0: sw -= 1
    if sw <= SMOOTH_POLY: sw = SMOOTH_POLY + 3 + (1 if (SMOOTH_POLY+3) % 2 == 0 else 0)
    ys = savgol_filter(y, window_length=sw, polyorder=min(SMOOTH_POLY, sw-1))
    pk, _ = find_peaks(ys, distance=BIG_PEAK_DIST,
                        prominence=BIG_PEAK_PROM, height=BIG_PEAK_HT)
    return pk

def extract_lr(sdf, foot_label):
    t0  = sdf['timestamp'].iloc[0]
    t   = (sdf['timestamp'] - t0).to_numpy() / 1000
    big = find_step_windows(sdf)
    recs = []
    for sensor in HEEL_SENSORS:
        y = pd.to_numeric(sdf[sensor], errors='coerce').to_numpy(dtype=float)
        for i in range(len(big) - 1):
            p1, p2 = big[i], big[i+1]
            ci  = p1 + int(np.argmin(y[p1:p2]))
            end = min(p2, ci + HILL_WINDOW)
            seg = y[ci:end+1]
            lp, _ = find_peaks(seg, prominence=1)
            hi  = ci + (lp[0] if len(lp) else int(np.argmax(seg[1:])) + 1)
            dt  = t[hi] - t[ci]
            dp  = y[hi] - y[ci]
            if dt <= 0 or dp < MIN_DP: continue
            recs.append({'foot': foot_label, 'sensor': sensor, 'window_id': i,
                         'time_s': round(float(t[ci]), 3),
                         'loading_rate': round(dp/dt, 2),
                         'pressure_rise': round(float(dp), 2),
                         'time_to_peak_ms': round(dt*1000, 2)})
    if not recs: return pd.DataFrame()
    all_res = pd.DataFrame(recs)
    avg = (all_res.groupby(['foot','window_id'], as_index=False)
           .agg(time_s=('time_s','mean'), loading_rate=('loading_rate','mean'),
                pressure_rise=('pressure_rise','mean'),
                time_to_peak_ms=('time_to_peak_ms','mean'),
                n_sensors=('sensor','count'))
           .pipe(lambda d: d[d['n_sensors'] == len(HEEL_SENSORS)])
           .drop(columns='window_id').reset_index(drop=True))
    avg['step'] = range(1, len(avg)+1)
    return avg

# ── Run ───────────────────────────────────────────────────────────────────────
ent_list, lr_list = [], []
for sid, label in [(1, 'Right'), (2, 'Left')]:
    sdf = raw[raw['sole_id'] == sid].sort_values('timestamp').reset_index(drop=True)
    ent_list.append(extract_entropy(sdf, label))
    lr_list.append(extract_lr(sdf, label))

ent = pd.concat(ent_list, ignore_index=True)
lr  = pd.concat(lr_list,  ignore_index=True)

for df_ref in [ent, lr]:
    df_ref.sort_values(['foot','time_s'], inplace=True)
    df_ref['step_n'] = df_ref.groupby('foot').cumcount() + 1
    df_ref.reset_index(drop=True, inplace=True)

for label, col in [('Evenness','evenness')]:
    for foot in ['Right','Left']:
        v = ent[ent.foot==foot][col]
        print(f'  Entropy {label} {foot:5s}: {v.mean():.4f} ± {v.std():.4f}  n={len(v)}')
for foot in ['Right','Left']:
    v = lr[lr.foot==foot]['loading_rate']
    print(f'  LR {foot:5s}: {v.mean():.0f} ± {v.std():.0f}  [{v.min():.0f}–{v.max():.0f}]  n={len(v)}')

  Entropy Evenness Right: 0.9875 ± 0.0043  n=340
  Entropy Evenness Left : 0.9906 ± 0.0023  n=290
  LR Right: 1196 ± 381  [240–2870]  n=337
  LR Left : 939 ± 347  [281–2125]  n=305


## Cell 4 — Figure-factory helper

In [4]:
def make_fig(title, subplot_titles, rows=1, cols=2,
             shared_y=False, height=480, specs=None):
    """Return a Plotly figure with a consistent 2-column layout."""
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=subplot_titles,
        shared_yaxes=shared_y,
        horizontal_spacing=0.10,
        specs=specs,
    )
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color='#333'),
                   x=0.5, xanchor='center'),
        height=height,
        **LAYOUT,
    )
    return fig


def hover_ent(row):
    return (f"<b>{row['foot']} foot</b><br>"
            f"Time: {row['time_s']:.1f} s<br>"
            f"Evenness: {row['evenness']:.5f}<br>"
            f"H: {row['H_bits']:.4f} bits<br>"
            f"Stance: {row['stance_ms']:.0f} ms")

def hover_lr(row):
    return (f"<b>{row['foot']} foot</b><br>"
            f"Time: {row['time_s']:.1f} s<br>"
            f"Loading rate: {row['loading_rate']:.0f} pressure/s<br>"
            f"Pressure rise: {row['pressure_rise']:.1f} ADC<br>"
            f"Time to peak: {row['time_to_peak_ms']:.1f} ms")

print('Figure factory ready')

Figure factory ready


## Cell 5 — Chart 1: Line Graph — Metric vs Walk Time

In [5]:
fig = make_fig(
    'Chart 1 — Line Graph: metric value over walk time',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)']
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    d = ent[ent.foot == foot].sort_values('time_s')
    fig.add_trace(go.Scatter(
        x=d['time_s'], y=d['evenness'],
        mode='lines+markers',
        name=f'{foot} foot',
        line=dict(color=color, width=1.8),
        marker=dict(size=5, color=color),
        legendgroup=foot, showlegend=show,
        hovertext=[hover_ent(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=1)

    d = lr[lr.foot == foot].sort_values('time_s')
    fig.add_trace(go.Scatter(
        x=d['time_s'], y=d['loading_rate'],
        mode='lines+markers',
        name=f'{foot} foot',
        line=dict(color=color, width=1.8),
        marker=dict(size=5, color=color),
        legendgroup=foot, showlegend=False,
        hovertext=[hover_lr(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=2)

fig.update_xaxes(title_text='Walk time (s)')
fig.update_yaxes(title_text='Evenness score (0–1)', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

## Cell 6 — Chart 2: Scatter Plot — Value vs Step Number

In [6]:
fig = make_fig(
    'Chart 2 — Scatter Plot: value vs step number',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)']
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    d = ent[ent.foot == foot]
    fig.add_trace(go.Scatter(
        x=d['step_n'], y=d['evenness'],
        mode='markers',
        name=f'{foot} foot',
        marker=dict(size=8, color=color, opacity=0.75,
                    line=dict(color='white', width=0.5)),
        legendgroup=foot, showlegend=show,
        hovertext=[hover_ent(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=1)

    d = lr[lr.foot == foot]
    fig.add_trace(go.Scatter(
        x=d['step_n'], y=d['loading_rate'],
        mode='markers',
        name=f'{foot} foot',
        marker=dict(size=8, color=color, opacity=0.75,
                    line=dict(color='white', width=0.5)),
        legendgroup=foot, showlegend=False,
        hovertext=[hover_lr(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=2)

fig.update_xaxes(title_text='Step number (per foot)')
fig.update_yaxes(title_text='Evenness score (0–1)', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

## Cell 7 — Chart 3: Interleaved Bar Chart — Per-Step Magnitude

In [7]:
fig = make_fig(
    'Chart 3 — Bar Chart: per-step magnitude (interleaved right / left)',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)'],
    height=520,
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    d = ent[ent.foot == foot].reset_index(drop=True)
    fig.add_trace(go.Bar(
        x=d['step_n'], y=d['evenness'],
        name=f'{foot} foot',
        marker_color=color, opacity=0.78,
        legendgroup=foot, showlegend=show,
        hovertext=[hover_ent(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=1)

    d = lr[lr.foot == foot].reset_index(drop=True)
    fig.add_trace(go.Bar(
        x=d['step_n'], y=d['loading_rate'],
        name=f'{foot} foot',
        marker_color=color, opacity=0.78,
        legendgroup=foot, showlegend=False,
        hovertext=[hover_lr(r) for _, r in d.iterrows()],
        hoverinfo='text',
    ), row=1, col=2)

# Mean lines
for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
    for foot, color, dash in [('Right', C_RIGHT, 'dash'), ('Left', C_LEFT, 'dot')]:
        mean_val = df_m[df_m.foot == foot][metric].mean()
        fig.add_hline(y=mean_val, line=dict(color=color, dash=dash, width=1.5),
                      annotation_text=f'{foot} mean {mean_val:.3g}',
                      annotation_font_size=10, row=1, col=col_n)

fig.update_layout(barmode='group')
fig.update_xaxes(title_text='Step number')
fig.update_yaxes(title_text='Evenness score (0–1)', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

## Cell 8 — Chart 4: Rolling Mean ± 1 SD Band

In [8]:
ROLL = 8

fig = make_fig(
    f'Chart 4 — Rolling Mean ± 1 SD  (window = {ROLL} steps)',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)']
)

import plotly.colors as pc

def hex_to_rgba(hex_col, alpha):
    h = hex_col.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)
    fill_color = hex_to_rgba(color, 0.15)

    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d  = df_m[df_m.foot == foot].sort_values('time_s').reset_index(drop=True)
        mn = d[metric].rolling(ROLL, center=True, min_periods=3).mean()
        sd = d[metric].rolling(ROLL, center=True, min_periods=3).std().fillna(0)

        # Raw dots
        fig.add_trace(go.Scatter(
            x=d['time_s'], y=d[metric],
            mode='markers',
            marker=dict(size=4, color=color, opacity=0.3),
            legendgroup=foot, showlegend=False,
            hoverinfo='skip',
        ), row=1, col=col_n)

        # SD band (upper then lower, reversed for fill)
        fig.add_trace(go.Scatter(
            x=pd.concat([d['time_s'], d['time_s'][::-1]]),
            y=pd.concat([mn + sd, (mn - sd)[::-1]]),
            fill='toself', fillcolor=fill_color,
            line=dict(color='rgba(0,0,0,0)'),
            legendgroup=foot, showlegend=False,
            hoverinfo='skip', name=f'{foot} ±SD',
        ), row=1, col=col_n)

        # Rolling mean line
        fig.add_trace(go.Scatter(
            x=d['time_s'], y=mn,
            mode='lines',
            name=f'{foot} foot',
            line=dict(color=color, width=2.5),
            legendgroup=foot, showlegend=(show and col_n == 1),
            hovertemplate=f'<b>{foot}</b><br>Time: %{{x:.1f}} s<br>Rolling mean: %{{y:.4f}}<extra></extra>',
        ), row=1, col=col_n)

fig.update_xaxes(title_text='Walk time (s)')
fig.update_yaxes(title_text='Evenness score', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

## Cell 9 — Chart 5: Box Plot + Raw Strip

In [9]:
fig = make_fig(
    'Chart 5 — Box Plot with raw data points overlaid',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)'],
    height=500,
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    for col_n, df_m, metric, hover_fn in [
        (1, ent, 'evenness',     hover_ent),
        (2, lr,  'loading_rate', hover_lr),
    ]:
        d = df_m[df_m.foot == foot]

        # Box
        fig.add_trace(go.Box(
            y=d[metric], name=f'{foot} foot',
            marker_color=color,
            fillcolor=hex_to_rgba(color, 0.4),
            line_color=color,
            boxmean='sd',
            notched=True,
            legendgroup=foot, showlegend=(show and col_n == 1),
            hovertemplate=f'<b>{foot}</b><br>%{{y:.4f}}<extra></extra>',
        ), row=1, col=col_n)

        # Jittered strip overlay
        rng = np.random.default_rng(42)
        jitter = rng.uniform(-0.2, 0.2, len(d))
        fig.add_trace(go.Scatter(
            x=[f'{foot} foot'] * len(d) + jitter,
            y=d[metric],
            mode='markers',
            marker=dict(size=5, color=color, opacity=0.5,
                        line=dict(color='white', width=0.3)),
            legendgroup=foot, showlegend=False,
            hovertext=[hover_fn(r) for _, r in d.iterrows()],
            hoverinfo='text',
        ), row=1, col=col_n)

# Mann-Whitney annotation
for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
    g1 = df_m[df_m.foot=='Right'][metric].values
    g2 = df_m[df_m.foot=='Left'][metric].values
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    sig  = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    fig.add_annotation(
        text=f'Mann-Whitney U  p={p:.4f} {sig}',
        xref=f'x{col_n} domain', yref=f'y{col_n} domain',
        x=0.5, y=1.08, showarrow=False,
        font=dict(size=10, color='#555'),
        row=1, col=col_n,
    )

fig.update_yaxes(title_text='Evenness score (0–1)', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U10'), dtype('float64')) -> None

## Cell 10 — Chart 6: Histogram

In [ ]:
fig = make_fig(
    'Chart 6 — Histogram: distribution shape and overlap',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)'],
    height=480,
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d = df_m[df_m.foot == foot][metric]

        fig.add_trace(go.Histogram(
            x=d, nbinsx=20,
            name=f'{foot} foot',
            marker_color=color, opacity=0.6,
            legendgroup=foot, showlegend=(show and col_n == 1),
            hovertemplate='%{x:.4f}: %{y} steps<extra></extra>',
        ), row=1, col=col_n)

        # Mean vertical line
        fig.add_vline(
            x=d.mean(), line=dict(color=color, dash='dash', width=1.8),
            annotation_text=f'{foot} mean', annotation_font_size=9,
            row=1, col=col_n,
        )

fig.update_layout(barmode='overlay')
fig.update_xaxes(title_text='Evenness score', col=1)
fig.update_xaxes(title_text='Loading rate (pressure/s)', col=2)
fig.update_yaxes(title_text='Count')
fig.show()

## Cell 11 — Chart 7: Cumulative Distribution (ECDF)

In [ ]:
fig = make_fig(
    'Chart 7 — Cumulative Distribution (ECDF)',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)']
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)
    fill_color = hex_to_rgba(color, 0.08)

    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        vals = np.sort(df_m[df_m.foot == foot][metric].values)
        cdf  = np.arange(1, len(vals)+1) / len(vals)

        # Filled area under ECDF
        fig.add_trace(go.Scatter(
            x=np.concatenate([vals, vals[::-1]]),
            y=np.concatenate([cdf, np.zeros(len(cdf))]),
            fill='toself', fillcolor=fill_color,
            line=dict(color='rgba(0,0,0,0)'),
            legendgroup=foot, showlegend=False,
            hoverinfo='skip',
        ), row=1, col=col_n)

        # ECDF step line
        fig.add_trace(go.Scatter(
            x=vals, y=cdf,
            mode='lines',
            name=f'{foot} foot',
            line=dict(color=color, width=2.5, shape='hv'),
            legendgroup=foot, showlegend=(show and col_n == 1),
            hovertemplate=f'<b>{foot}</b><br>Value: %{{x:.4f}}<br>Cumulative: %{{y:.2f}}<extra></extra>',
        ), row=1, col=col_n)

fig.update_xaxes(title_text='Evenness score', col=1)
fig.update_xaxes(title_text='Loading rate (pressure/s)', col=2)
fig.update_yaxes(title_text='Cumulative proportion', range=[0, 1.05])
fig.show()

## Cell 12 — Chart 8: Left–Right Symmetry per Matched Step Pair

In [ ]:
fig = make_fig(
    'Chart 8 — Left–Right Symmetry: |Right − Left| per matched step pair',
    ['Evenness |R − L|', 'Loading Rate |R − L|']
)

SYM_COLOR = '#9467BD'

for col_n, df_m, metric, ylabel in [
    (1, ent, 'evenness',     '|R−L| evenness'),
    (2, lr,  'loading_rate', '|R−L| loading rate (pressure/s)'),
]:
    r = df_m[df_m.foot=='Right'].sort_values('time_s').reset_index(drop=True)
    l = df_m[df_m.foot=='Left'].sort_values('time_s').reset_index(drop=True)

    pairs = []
    for _, rrow in r.iterrows():
        idx = (l['time_s'] - rrow['time_s']).abs().idxmin()
        pairs.append({
            'pair': len(pairs), 'time_s': rrow['time_s'],
            'diff': abs(rrow[metric] - l.loc[idx, metric]),
            'right_val': rrow[metric], 'left_val': l.loc[idx, metric],
        })
    pairs_df = pd.DataFrame(pairs)
    mean_diff = pairs_df['diff'].mean()

    fig.add_trace(go.Bar(
        x=pairs_df['pair'], y=pairs_df['diff'],
        name='|R−L|',
        marker_color=SYM_COLOR, opacity=0.75,
        legendgroup='sym', showlegend=(col_n == 1),
        hovertemplate=(
            '<b>Step pair %{x}</b><br>'
            f'Right: %{{customdata[0]:.4f}}<br>'
            f'Left:  %{{customdata[1]:.4f}}<br>'
            'Diff:  %{y:.4f}<extra></extra>'
        ),
        customdata=pairs_df[['right_val','left_val']].values,
    ), row=1, col=col_n)

    fig.add_hline(
        y=mean_diff, line=dict(color='black', dash='dash', width=1.5),
        annotation_text=f'Mean diff = {mean_diff:.4g}',
        annotation_font_size=10, row=1, col=col_n,
    )

fig.update_xaxes(title_text='Matched step pair index')
fig.update_yaxes(title_text='|R−L| evenness', col=1)
fig.update_yaxes(title_text='|R−L| loading rate (pressure/s)', col=2)
fig.show()

## Cell 13 — Chart 9: Violin Plot

In [ ]:
fig = make_fig(
    'Chart 9 — Violin Plot: distribution shape, density and spread',
    ['Midstance Evenness Score', 'Heel-Strike Loading Rate (pressure/s)'],
    height=500,
)

shown = set()
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    show = foot not in shown; shown.add(foot)

    for col_n, df_m, metric, hover_fn in [
        (1, ent, 'evenness',     hover_ent),
        (2, lr,  'loading_rate', hover_lr),
    ]:
        d = df_m[df_m.foot == foot]

        fig.add_trace(go.Violin(
            y=d[metric], name=f'{foot} foot',
            box_visible=True,
            meanline_visible=True,
            points='all',
            jitter=0.3,
            pointpos=0,
            fillcolor=hex_to_rgba(color, 0.45),
            line_color=color,
            marker=dict(color=color, size=4, opacity=0.5),
            legendgroup=foot, showlegend=(show and col_n == 1),
            hovertext=[hover_fn(r) for _, r in d.iterrows()],
            hoverinfo='text',
        ), row=1, col=col_n)

fig.update_layout(violinmode='group')
fig.update_yaxes(title_text='Evenness score (0–1)', col=1)
fig.update_yaxes(title_text='Loading rate (pressure/s)', col=2)
fig.show()

## Cell 14 — Chart 10: Cross-Metric Scatter — Entropy vs Loading Rate

In [ ]:
fig = go.Figure()
fig.update_layout(
    title=dict(text='Chart 10 — Evenness vs Loading Rate per Step<br>'
               '<sup>Does a harder heel strike correlate with more uneven pressure distribution?</sup>',
               font=dict(size=14), x=0.5, xanchor='center'),
    xaxis_title='Loading Rate (pressure/s)',
    yaxis_title='Evenness Score (0–1)',
    height=520,
    **LAYOUT,
)

for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    e = ent[ent.foot == foot].sort_values('time_s').reset_index(drop=True)
    l = lr[lr.foot  == foot].sort_values('time_s').reset_index(drop=True)
    merged = pd.merge_asof(e[['time_s','evenness','H_bits','stance_ms']],
                           l[['time_s','loading_rate','pressure_rise','time_to_peak_ms']],
                           on='time_s', direction='nearest', tolerance=2.0)
    merged = merged.dropna()

    fig.add_trace(go.Scatter(
        x=merged['loading_rate'], y=merged['evenness'],
        mode='markers',
        name=f'{foot} foot',
        marker=dict(size=9, color=color, opacity=0.75,
                    line=dict(color='white', width=0.5)),
        hovertemplate=(
            f'<b>{foot}</b><br>'
            'LR: %{x:.0f} pressure/s<br>'
            'Evenness: %{y:.5f}<br>'
            'H: %{customdata[0]:.4f} bits<br>'
            'Stance: %{customdata[1]:.0f} ms<extra></extra>'
        ),
        customdata=merged[['H_bits','stance_ms']].values,
    ))

    # OLS trend line
    if len(merged) > 2:
        z  = np.polyfit(merged['loading_rate'], merged['evenness'], 1)
        xr = np.linspace(merged['loading_rate'].min(), merged['loading_rate'].max(), 100)
        fig.add_trace(go.Scatter(
            x=xr, y=np.polyval(z, xr),
            mode='lines',
            name=f'{foot} trend',
            line=dict(color=color, width=1.5, dash='dash'),
            hoverinfo='skip',
            legendgroup=foot,
        ))

fig.show()

## Cell 15 — Combined Dashboard: all 10 charts in one tall figure

Scroll through all charts in a single interactive figure. Each row is one chart type.

In [ ]:
from plotly.subplots import make_subplots

ROWS = 5
row_titles = [
    'Line graph — metric over time',
    'Rolling mean ± 1 SD band',
    'Box plot + raw data points',
    'Histogram — distribution shape',
    'Symmetry — |Right − Left| per matched pair',
]
subplot_titles = []
for rt in row_titles:
    subplot_titles += [f'Evenness  |  {rt}', f'Loading Rate  |  {rt}']

dashboard = make_subplots(
    rows=ROWS, cols=2,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.10,
    vertical_spacing=0.06,
)
dashboard.update_layout(
    title=dict(text='Gait Analysis Dashboard — Midstance Entropy & Loading Rate',
               font=dict(size=16), x=0.5, xanchor='center'),
    height=ROWS * 320,
    template='plotly_white',
    font=dict(family='Arial', size=10),
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.01,
                xanchor='right', x=1),
    margin=dict(t=100, b=50),
)

# ── Helper to add a trace only once to the legend ────────────────────────────
legend_shown = set()

def add(trace, row, col, foot=None):
    key = (foot, row)  # show legend once per foot per row
    if foot and key not in legend_shown:
        trace.showlegend = True
        legend_shown.add(key)
    else:
        trace.showlegend = False
    dashboard.add_trace(trace, row=row, col=col)

# ── Row 1: Line graphs ────────────────────────────────────────────────────────
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d = df_m[df_m.foot == foot].sort_values('time_s')
        add(go.Scatter(x=d['time_s'], y=d[metric], mode='lines+markers',
                       name=f'{foot} foot', legendgroup=foot,
                       line=dict(color=color, width=1.5),
                       marker=dict(size=4, color=color),
                       hovertemplate=f'<b>{foot}</b> %{{x:.1f}}s: %{{y:.4f}}<extra></extra>'),
            row=1, col=col_n, foot=foot)

# ── Row 2: Rolling mean ± SD ──────────────────────────────────────────────────
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    fill_color = hex_to_rgba(color, 0.12)
    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d  = df_m[df_m.foot == foot].sort_values('time_s').reset_index(drop=True)
        mn = d[metric].rolling(ROLL, center=True, min_periods=3).mean()
        sd = d[metric].rolling(ROLL, center=True, min_periods=3).std().fillna(0)
        # SD band
        add(go.Scatter(
            x=pd.concat([d['time_s'], d['time_s'][::-1]]),
            y=pd.concat([mn+sd, (mn-sd)[::-1]]),
            fill='toself', fillcolor=fill_color,
            line=dict(color='rgba(0,0,0,0)'),
            hoverinfo='skip', legendgroup=foot),
            row=2, col=col_n)
        # Mean line
        add(go.Scatter(x=d['time_s'], y=mn, mode='lines',
                       name=f'{foot} foot', legendgroup=foot,
                       line=dict(color=color, width=2),
                       hovertemplate=f'<b>{foot}</b> %{{x:.1f}}s: %{{y:.4f}}<extra></extra>'),
            row=2, col=col_n, foot=foot)

# ── Row 3: Box plots ──────────────────────────────────────────────────────────
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d = df_m[df_m.foot == foot]
        add(go.Box(y=d[metric], name=f'{foot} foot',
                   marker_color=color,
                   fillcolor=hex_to_rgba(color, 0.35),
                   line_color=color, boxmean='sd', notched=True,
                   legendgroup=foot,
                   hovertemplate=f'<b>{foot}</b><br>%{{y:.4f}}<extra></extra>'),
            row=3, col=col_n, foot=foot)

# ── Row 4: Histograms ─────────────────────────────────────────────────────────
for foot, color in [('Right', C_RIGHT), ('Left', C_LEFT)]:
    for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
        d = df_m[df_m.foot == foot][metric]
        add(go.Histogram(x=d, nbinsx=18, name=f'{foot} foot',
                         marker_color=color, opacity=0.6,
                         legendgroup=foot,
                         hovertemplate='%{x:.4f}: %{y} steps<extra></extra>'),
            row=4, col=col_n, foot=foot)

dashboard.update_layout(barmode='overlay', **{k:v for k,v in
    dict(bargroupgap=0.1).items()})

# ── Row 5: Symmetry bars ──────────────────────────────────────────────────────
for col_n, df_m, metric in [(1, ent, 'evenness'), (2, lr, 'loading_rate')]:
    r = df_m[df_m.foot=='Right'].sort_values('time_s').reset_index(drop=True)
    l = df_m[df_m.foot=='Left'].sort_values('time_s').reset_index(drop=True)
    pairs = []
    for _, rrow in r.iterrows():
        idx = (l['time_s'] - rrow['time_s']).abs().idxmin()
        pairs.append(abs(rrow[metric] - l.loc[idx, metric]))
    add(go.Bar(x=list(range(len(pairs))), y=pairs,
               name='|R−L|', marker_color='#9467BD', opacity=0.75,
               legendgroup='sym',
               hovertemplate='Pair %{x}: |R−L| = %{y:.4f}<extra></extra>'),
        row=5, col=col_n)
    dashboard.add_hline(y=np.mean(pairs), line=dict(color='black', dash='dash', width=1.2),
                        row=5, col=col_n)

# ── Axis labels ───────────────────────────────────────────────────────────────
dashboard.update_yaxes(title_text='Evenness', col=1)
dashboard.update_yaxes(title_text='LR (pressure/s)', col=2)
for r in [1, 2]:
    dashboard.update_xaxes(title_text='Walk time (s)', row=r)
for r in [3, 4]:
    dashboard.update_xaxes(title_text='', row=r)
dashboard.update_xaxes(title_text='Step pair index', row=5)

dashboard.show()